# 05b: inference

Turns the test DICOMs into submission.csv. Runs with Internet off.

MODE = "model": average the fold checkpoints from 05a over every study.
MODE = "prior": every study gets 0.5 (pipeline smoke test).
A study that cannot be decoded or predicted falls back to 0.5, and a prior submission is
written first, so a later failure still leaves a valid file.

Inputs: competition data, 00-dicom-decoders output, 05a output (best_f*.pt),
utility scripts knee_prep and knee_model.
Settings: GPU T4 x2 (one is used), Internet off, Environment pinned to original.

# Cell 1: imports, prior submission, decoders, models

In [ ]:
import os, sys, glob, time, subprocess
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
SPLIT = "test_series"
MODE = "model"
T0 = time.time()

def find_all(pattern):
    hits = glob.glob(f"/kaggle/usr/lib/**/{pattern}", recursive=True)
    for top in os.listdir("/kaggle/input"):
        if top != "competitions":
            hits += glob.glob(f"/kaggle/input/{top}/**/{pattern}", recursive=True)
    return sorted(hits)

for name in ["knee_prep", "knee_model"]:
    hits = find_all(f"{name}.py")
    print(name, hits)
    if not hits:
        raise FileNotFoundError(f"{name}.py not attached")
    sys.path.insert(0, os.path.dirname(hits[0]))
import knee_prep as kp
import knee_model as km

test = pd.read_csv(f"{ROOT}/test.csv")
sample = pd.read_csv(f"{ROOT}/sample_submission.csv")
PRIOR = np.full(len(km.LABELS), 0.5)
km.submission_frame(test["StudyInstanceUID"], [PRIOR] * len(test)).to_csv("/kaggle/working/submission.csv", index=False)
print("prior submission written for", len(test), "studies")

wheels = [d for d in find_all("wheels") if os.path.isdir(d)]
if wheels:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", wheels[0],
                        "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "pylibjpeg-rle",
                        "python-gdcm"], capture_output=True, text=True)
    print("decoder install return code:", r.returncode, r.stderr[-300:])
else:
    print("decoder wheels not found")

MODELS, DEVICE = [], "cpu"
if MODE == "model":
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    ckpts = find_all("best_f*.pt")
    print("checkpoints:", [os.path.basename(c) for c in ckpts])
    try:
        MODELS = km.load_models(ckpts, DEVICE)
    except Exception as err:
        print("model load failed:", repr(err)[:300])
    print(len(MODELS), "models on", DEVICE)

# Cell 2: scan test series and laterality

In [ ]:
inv = None
try:
    ser = pd.read_csv(f"{ROOT}/test_series.csv")
    inv = kp.scan_all(ser, ROOT, SPLIT)
    lat = kp.study_laterality(inv)
    print(f"{len(test)} studies, {len(ser)} series, scan errors {inv['error'].notna().sum()}, "
          f"elapsed {time.time() - T0:.0f}s")
    print(lat["side_src"].value_counts().to_string())
except Exception as err:
    print("scan failed:", repr(err)[:300])

# Cell 3: decode, resample and predict every study

In [ ]:
preds, log = {}, []
if inv is not None:
    try:
        for sid, vols, notes, sec in kp.iter_studies(inv, ROOT, SPLIT, workers=4):
            p = PRIOR
            if MODELS and vols:
                try:
                    p = km.predict_study(MODELS, vols, DEVICE)
                except Exception as err:
                    notes = notes + [f"predict: {type(err).__name__}"]
            preds[sid] = p
            log.append({"StudyInstanceUID": sid, "planes": "".join(sorted(k[0] for k in vols)),
                        "n_planes": len(vols), "notes": "; ".join(notes), "sec": round(sec, 2)})
    except Exception as err:
        print("pipeline stopped:", repr(err)[:300])

log = pd.DataFrame(log, columns=["StudyInstanceUID", "planes", "n_planes", "notes", "sec"])
el = time.time() - T0
print("studies processed:", len(log), "of", len(test))
print(log["n_planes"].value_counts().to_string())
print("studies with fallback notes:", int((log["notes"] != "").sum()))
for n in log.loc[log["notes"] != "", "notes"].head(10):
    print("  ", n)
print(f"mean {log['sec'].mean():.2f}s per study per worker, total elapsed {el:.0f}s")

# Cell 4: write and check submission.csv

In [ ]:
probs = [preds.get(sid, PRIOR) for sid in test["StudyInstanceUID"]]
sub = km.submission_frame(test["StudyInstanceUID"], probs)
problems = km.check_submission(sub, sample, test["StudyInstanceUID"])
print("problems:", problems or "none")
if not problems:
    sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.round(3).head().to_string())
print(f"total {time.time() - T0:.0f}s")